In [1]:
# Setup
import time, os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv("../.env")
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Same test input for ALL prompts 
test_status   = "Fault"
test_temp     = 108
test_vib      = 12.0
test_pressure = 6.1
test_sound    = 92
test_hours    = 7800
test_type     = "Motor"
test_id       = "M005"

def call_llm(system_prompt, user_prompt, model="gpt-4o-mini"):
    start = time.time()
    r = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        max_tokens=350,
        temperature=0.4
    )
    return {
        "text":    r.choices[0].message.content,
        "latency": round(time.time() - start, 2),
        "tokens":  r.usage.total_tokens
    }

In [2]:
# Define all prompt versions to compare

#  P01- Current Prompt
system_initial = """You are an industrial maintenance engineer.
Provide a clear explanation of the issue and suggest practical maintenance actions.

Format:
Issue:
Cause:
Recommendation:"""

user_initial = f"""Machine status: {test_status}
Temperature: {test_temp}°C, Vibration: {test_vib} mm/s, Pressure: {test_pressure} bar, Sound: {test_sound} dB, Running hours: {test_hours}
Explain the likely issue and recommend one immediate action."""

# P02 — Role + context, natural paragraphs
system_p02 = """You are a senior industrial maintenance engineer with 20 years experience.
Explain what is wrong with the machine, why it is happening, 
and what the engineer should do immediately.
Write in natural paragraphs. Be specific about sensor values. Max 4 sentences."""

user_p02 = f"""Machine status: {test_status}
Temperature: {test_temp}°C, Vibration: {test_vib} mm/s, 
Pressure: {test_pressure} bar, Sound: {test_sound} dB, 
Running hours: {test_hours}"""

# P03 — Chain of thought + machine context
system_p03 = """You are a senior industrial maintenance engineer with 20 years experience 
in rotating machinery including motors, pumps, compressors, fans and conveyors.

When analyzing machine sensor data always cover:
1. Which sensor is abnormal and how severe
2. Most likely root cause
3. One immediate action right now
4. One preventive recommendation

Write in natural paragraphs. Be specific about sensor values. Max 5 sentences."""

user_p03 = f"""A {test_type} (ID: {test_id}) has been flagged as {test_status}.

Sensor readings:
- Temperature:   {test_temp}°C
- Vibration:     {test_vib} mm/s
- Pressure:      {test_pressure} bar
- Sound level:   {test_sound} dB
- Running hours: {test_hours} hrs

Analyze and advise."""

In [3]:
# Compare
versions = {
    "P01 — Current ": (system_initial, user_initial),
    "P02 — Role + natural":            (system_p02,      user_p02),
    "P03 — Chain of thought + context":(system_p03,      user_p03),
}

comparison_results = {}

for name, (sys_p, usr_p) in versions.items():
    r = call_llm(sys_p, usr_p)
    comparison_results[name] = r
    print(f"\n{'='*60}")
    print(f"VERSION: {name}")
    print(f"Latency: {r['latency']}s | Tokens: {r['tokens']}")
    print(f"\nResponse:\n{r['text']}")


VERSION: P01 — Current 
Latency: 4.58s | Tokens: 305

Response:
Issue: 
The machine is exhibiting signs of overheating, excessive vibration, high noise levels, and abnormal pressure readings, indicating potential mechanical failure or operational inefficiencies.

Cause:
The high temperature of 108°C suggests that the machine may be experiencing inadequate cooling or lubrication. The vibration level of 12.0 mm/s is above the acceptable threshold, which could indicate misalignment, imbalance, or wear in bearings or other rotating components. The sound level of 92 dB indicates that the machine is operating under stress, possibly due to the aforementioned issues. The pressure reading of 6.1 bar may be abnormal depending on the machine's specifications, indicating a possible blockage or failure in the pressure regulation system.

Recommendation:
As an immediate action, shut down the machine to prevent further damage and allow it to cool down. Once the machine is safely powered down, conduc

In [4]:
# Summary table
print("\n" + "="*60)
print(f"{'Version':<40} {'Latency':>8} {'Tokens':>8}")
print("="*60)
for name, r in comparison_results.items():
    print(f"{name:<40} {r['latency']:>7}s {r['tokens']:>8}")
print("="*60)


Version                                   Latency   Tokens
P01 — Current                               4.58s      305
P02 — Role + natural                         3.1s      235
P03 — Chain of thought + context            3.82s      339


In [ ]:
# GPT-3.5-turbo vs GPT-4o-mini using winning P02 prompt
print("Model Comparison — P02 prompt on both models\n")

model_results = {}

for model in ["gpt-3.5-turbo", "gpt-4o-mini"]:
    r = call_llm(system_p02, user_p02, model=model)
    cost = r["tokens"] * (0.000002 if "3.5" in model else 0.0000006)
    model_results[model] = {**r, "cost": cost}
    print(f"{'='*60}")
    print(f"Model: {model}")
    print(f"Latency: {r['latency']}s | Tokens: {r['tokens']} | Est. cost: ${cost:.5f}")
    print(f"\nResponse:\n{r['text']}")

# Summary
print(f"\n{'='*60}")
print(f"{'Model':<20} {'Latency':>9} {'Tokens':>8} {'Cost/call':>12}")
print(f"{'='*60}")
for m, r in model_results.items():
    print(f"{m:<20} {r['latency']:>8}s {r['tokens']:>8} ${r['cost']:>11.5f}")
print(f"{'='*60}")

Model Comparison — P02 prompt on both models

Model: gpt-3.5-turbo
Latency: 2.13s | Tokens: 219 | Est. cost: $0.00044

Response:
The machine is currently experiencing high temperature, vibration, and sound levels, indicating a potential issue with overheating and mechanical stress. These conditions can lead to accelerated wear and tear on components, ultimately resulting in machine failure if not addressed promptly. The engineer should immediately shut down the machine to prevent further damage, inspect the cooling system for blockages or malfunctions, and check the alignment and condition of rotating parts for potential imbalance or misalignment. Additionally, it is crucial to review the maintenance history and schedule a thorough inspection to identify any underlying issues causing the abnormal sensor values.
Model: gpt-4o-mini
Latency: 2.33s | Tokens: 248 | Est. cost: $0.00015

Response:
The machine is experiencing a fault likely due to overheating, as indicated by the elevated temp

In [11]:
# 5 test cases faithfulness evaluation
test_cases = [
    {"status": "Fault",   "temp": 115, "vib": 12.4,
     "pressure": 6.2, "sound": 94,  "hours": 7800,
     "type": "Motor",    "id": "M005",
     "themes": ["vibration", "bearing", "overheating", "shutdown"]},

    {"status": "Warning", "temp": 88,  "vib": 5.2,
     "pressure": 4.8, "sound": 77,  "hours": 5100,
     "type": "Pump",     "id": "M010",
     "themes": ["temperature", "inspection", "monitor", "schedule"]},

    {"status": "Normal",  "temp": 52,  "vib": 1.8,
     "pressure": 3.2, "sound": 61,  "hours": 1200,
     "type": "Fan",      "id": "M003",
     "themes": ["normal", "routine", "no immediate", "good"]},

    {"status": "Fault",   "temp": 125, "vib": 14.0,
     "pressure": 7.0, "sound": 100, "hours": 9500,
     "type": "Compressor","id": "M015",
     "themes": ["critical", "immediate", "shutdown", "failure"]},

    {"status": "Warning", "temp": 78,  "vib": 4.5,
     "pressure": 4.5, "sound": 74,  "hours": 4200,
     "type": "Conveyor",  "id": "M008",
     "themes": ["warning", "pressure", "check", "maintenance"]},
]

print("="*60)
print("Faithfulness Evaluation — 5 Test Cases (P02 + gpt-4o-mini)")
print("="*60)

total = 0
for i, tc in enumerate(test_cases):
    usr = f"""A {tc['type']} (ID: {tc['id']}) has been flagged as {tc['status']}.
Sensor readings:
- Temperature:   {tc['temp']}°C
- Vibration:     {tc['vib']} mm/s
- Pressure:      {tc['pressure']} bar
- Sound level:   {tc['sound']} dB
- Running hours: {tc['hours']} hrs
Analyze and advise."""

    r = call_llm(system_p03, usr, model="gpt-4o-mini")
    matched = [t for t in tc["themes"] if t in r["text"].lower()]
    score   = len(matched) / len(tc["themes"])
    total  += score

    print(f"\nCase {i+1} — {tc['type']} {tc['id']} | Status: {tc['status']}")
    print(f"Expected themes : {tc['themes']}")
    print(f"Matched         : {matched}")
    print(f"Score           : {len(matched)}/{len(tc['themes'])} = {score:.0%}")

avg = total / len(test_cases)
print(f"\n{'='*60}")
print(f"Overall Faithfulness Score: {avg:.0%}")
print(f"{'='*60}")

Faithfulness Evaluation — 5 Test Cases (P02 + gpt-4o-mini)

Case 1 — Motor M005 | Status: Fault
Expected themes : ['vibration', 'bearing', 'overheating', 'shutdown']
Matched         : ['vibration', 'overheating']
Score           : 2/4 = 50%

Case 2 — Pump M010 | Status: Warning
Expected themes : ['temperature', 'inspection', 'monitor', 'schedule']
Matched         : ['temperature', 'inspection', 'monitor', 'schedule']
Score           : 4/4 = 100%

Case 3 — Fan M003 | Status: Normal
Expected themes : ['normal', 'routine', 'no immediate', 'good']
Matched         : ['normal', 'routine']
Score           : 2/4 = 50%

Case 4 — Compressor M015 | Status: Fault
Expected themes : ['critical', 'immediate', 'shutdown', 'failure']
Matched         : ['immediate']
Score           : 1/4 = 25%

Case 5 — Conveyor M008 | Status: Warning
Expected themes : ['warning', 'pressure', 'check', 'maintenance']
Matched         : ['warning', 'check', 'maintenance']
Score           : 3/4 = 75%

Overall Faithfulness S